# Kobol Expander VCF (SSM2040): neural grey-box, full pipeline
Dataset: `../Kobol_VCO_static` (Á. Crespo 2024). Despite the name it is the **filter** dataset: 4 VCO notes × 8 resonance × 8 cutoff.
Runs on a laptop CPU in about 15 min, NumPy/SciPy/matplotlib only. The C++ DSP in `../dsp` is compiled automatically by `dsp.py` (needs a C++ compiler). All plots are written to `figs/` (PNG) and `../paper-latex/figs/` (PDF).

In [ ]:
import numpy as np, matplotlib
print(np.__version__, matplotlib.__version__)

## 0. The data
Cutoff sweep, resonance sweep, and where the filter self-oscillates (resonance ≥ 111).

In [ ]:
!python extract.py

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_data.png'))

## 1. Linear analysis: is it really a 4-pole filter?
Joint fit of an unknown per-note source spectrum and `g / ((1 + s/wc)^N + K)` on the 192 non-oscillating files, for N = 2, 3, 4, 6.

In [ ]:
for N in (2, 3, 4, 6):
    !python fit_linear.py {N} | head -1

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_linfit.png'))

### Reconstructed filter input (the VCO saw)
Same for all four notes. This gives the aligned input that the thesis' PedalNet never had.

In [ ]:
!python source.py

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_source.png'))

## 2. Nonlinear model on the self-oscillating files
Four tanh OTA cells and a saturating CA3080 in the resonance loop (`../dsp/kobol_vcf.h`). Global drive and feedback knee by grid search, then per-file fc, K, gain by simulation.

In [ ]:
!python fit_osc.py refine
!python fit_osc.py full 0.1 18 | tail -5

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_nonlinear.png'))

## 3. Control law: physical + neural residual
Targets per setting, then cross-validation of four laws (pure MLP, physical, interpolation, hybrid) and the final hybrid fit.

In [ ]:
!rm -f figs/cache_*.npy   # plots cache heavy steps; clear them after re-fitting
!python law.py cv final | tail -16

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_params.png'))
display(Image('figs/fig_vcf_lawcv.png'))

### Training curves
Train (solid) and held-out (dashed) loss of the pure MLP and of the hybrid's residual network, five seeds each.

In [ ]:
!python figs.py training

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_training.png'))

## 4. End-to-end evaluation against the recordings
Law trained without the held-out setting → C++ filter → compare with the recording (harmonics, waveform ESR, oscillation).

In [ ]:
!python eval_audio.py

In [ ]:
!python figs.py data source linfit params nonlinear lawcv eval cycles system

In [ ]:
from IPython.display import Image, display
display(Image('figs/fig_vcf_eval.png'))
display(Image('figs/fig_vcf_cycles.png'))

## 5. Listen: recording vs model at a held-out setting

In [ ]:
from IPython.display import Audio, display
from listen import ab
rec, mod, sr = ab('B3', 63, 79)        # note, resonance step, cutoff step (any of 15, 31, ..., 127)
display(Audio(rec, rate=sr)); display(Audio(mod, rate=sr))

## 6. Export for the plugin and the browser prototype

In [ ]:
!python export_law.py
!python export_js.py